In [0]:
from pyspark.sql import functions as F

source_path = "/Volumes/roadsafe/bronze/raw_files/"
checkpoint  = "/Volumes/roadsafe/bronze/raw_files/_checkpoints/accidents_v3/"

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("sep", ",")
        .option("cloudFiles.inferColumnTypes", "false")   # keep all as text
        .option("cloudFiles.schemaLocation", checkpoint + "schema")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("pathGlobFilter", "unfallatlas*.csv")
        .load(source_path)
        .withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
)

(    df.writeStream
      .option("checkpointLocation", checkpoint + "write")
      .option("mergeSchema", "true")
      .trigger(availableNow=True)               # process everything, then stop
      .toTable("roadsafe.bronze.accidents")
      .awaitTermination()                      # wait for the batch to finish
)

In [0]:
%sql
SELECT _source_file, COUNT(*) AS rows
FROM roadsafe.bronze.accidents
GROUP BY _source_file ORDER BY _source_file;

In [0]:
spark.table("roadsafe.bronze.accidents").printSchema()